In [1]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.linear_model import LogisticRegression
import re

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from datasets import Dataset


In [2]:
comment_df = pd.read_csv('comments.csv')

In [3]:
comment_df.head()

,id,product_id,content,customer_id,rating
0,20227442,279218500,Great,11471633,5
1,20227443,279218500,Great,11471633,5
2,20227444,279218500,Great,11471633,5
3,20227445,279218500,Great,11471633,5
4,20211361,279112904,"Hay. Mê truyện từ còn bé, không thể từ bỏ được",29833486,5


In [4]:
comment_df.shape

(11335, 5)

In [5]:
product_df = pd.read_csv('product.csv')
product_df.head()

,id,name
0,279218500,Ship of Theseus - NXB Trẻ
1,279112904,Thám Tử Lừng Danh Conan - Tập 107
2,278997061,Sách Tô Màu Phát Triển Trí Não Bộ Cho Bé 1-5 T...
3,278967562,Monster - Deluxe Edition – [Chọn Tập Lẻ
4,278856619,Sách Cô Bé Hàng Xóm Và Bốn Viên Kẹo (Nguyễn Nh...


In [47]:
df = comment_df.merge(product_df, left_on='product_id', right_on='id')
df.head()

,id_x,product_id,content,customer_id,rating,id_y,name
0,20227442,279218500,Great,11471633,5,279218500,Ship of Theseus - NXB Trẻ
1,20227443,279218500,Great,11471633,5,279218500,Ship of Theseus - NXB Trẻ
2,20227444,279218500,Great,11471633,5,279218500,Ship of Theseus - NXB Trẻ
3,20227445,279218500,Great,11471633,5,279218500,Ship of Theseus - NXB Trẻ
4,20211361,279112904,"Hay. Mê truyện từ còn bé, không thể từ bỏ được",29833486,5,279112904,Thám Tử Lừng Danh Conan - Tập 107


In [48]:
df['rating'].value_counts()

,count
rating,
5,9505
4,1071
3,339
1,253
2,159


In [7]:
df = df.rename(columns={"rating":"labels"})

In [8]:
df = df.drop(['id_y'], axis=1)
df = df.dropna(subset=['content', 'labels'])

In [9]:
df = df[df['labels'] != 3]
df['labels'] = df['labels'].apply(lambda  x:1 if x > 3 else 0)

In [45]:
df['labels'].value_counts()

,count
labels,
1,10576
0,412


In [10]:
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['labels'])
test_df, val_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['labels'])

In [11]:
train_df = Dataset.from_pandas(train_df)
test_df = Dataset.from_pandas(test_df)
val_df = Dataset.from_pandas(val_df)

In [12]:
model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

In [13]:
def tokenize(row):
  return tokenizer(row['content'], truncation=True, padding='max_length', max_length=256)

tokenize_train = train_df.map(tokenize, batched=True)
tokenize_test = test_df.map(tokenize, batched=True)
tokenize_val = val_df.map(tokenize, batched=True)

Map:   0%|          | 0/8790 [00:00<?, ? examples/s]

Map:   0%|          | 0/1099 [00:00<?, ? examples/s]

Map:   0%|          | 0/1099 [00:00<?, ? examples/s]

In [14]:
print(tokenize_train)

Dataset({
    features: ['id_x', 'product_id', 'content', 'customer_id', 'labels', 'name', '__index_level_0__', 'input_ids', 'attention_mask'],
    num_rows: 8790
})


In [15]:
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [16]:
from sklearn.metrics import f1_score, accuracy_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    f1  = f1_score(labels, preds)
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1
    }

In [17]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    metric_for_best_model='f1',
    load_best_model_at_end=True,
)

In [18]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenize_train,
    eval_dataset=tokenize_test,
    compute_metrics=compute_metrics
)

In [19]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.212488,0.183068,0.962693,0.980992
2,0.165880,0.140642,0.962693,0.980992
3,0.144284,0.120442,0.963603,0.981378


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=1650, training_loss=0.17050419778534862, metrics={'train_runtime': 1586.6965, 'train_samples_per_second': 16.619, 'train_steps_per_second': 1.04, 'total_flos': 3469119264921600.0, 'train_loss': 0.17050419778534862, 'epoch': 3.0})

In [20]:
test_results = trainer.evaluate(tokenize_test)
print(test_results)

{'eval_loss': 0.12044225633144379, 'eval_accuracy': 0.9636032757051866, 'eval_f1': 0.9813780260707635, 'eval_runtime': 17.781, 'eval_samples_per_second': 61.808, 'eval_steps_per_second': 1.012, 'epoch': 3.0}


In [23]:
df = Dataset.from_pandas(df)
tokenized_df = df.map(tokenize, batched=True)

Map:   0%|          | 0/10988 [00:00<?, ? examples/s]

In [25]:
predicted_labels = trainer.predict(tokenized_df)

In [31]:
df['predicted'] = predicted_labels.predictions.argmax(axis=1)
product = df.groupby(['product_id', 'name']).agg(
    total = ('content', 'count'),
    positive = ('predicted', 'sum')
).reset_index()

In [34]:
product.shape

(80, 4)

In [35]:
product['negative'] = product['total'] - product['positive']
product['positive_rate'] = round((product['positive'] / product['total']) * 100, 2)

In [40]:
product_stats = product.sort_values(by='negative', ascending=False)

In [43]:
product_stats.head(20)

,product_id,name,total,positive,negative,positive_rate
11,74021317,Sách Cây Cam Ngọt Của Tôi,4622,4575,47,98.98
1,1080002,Sách Boxset Harry Potter - Tiếng Việt (Trọn Bộ...,853,840,13,98.48
4,7833728,Sách Dạy Con Làm Giàu (Tập 1) - Để Không Có Ti...,1286,1273,13,98.99
5,7982628,Sách Suối Nguồn (Tái Bản),671,662,9,98.66
6,13446508,Sách Cánh Đồng Bất Tận (Tái Bản),293,288,5,98.29
0,540040,Sách Giáo Trình Chuẩn HSK 1 - Bài Tập (Kèm fil...,200,197,3,98.50
9,32490132,Sách Một Đời Như Kẻ Tìm Đường,328,325,3,99.09
12,111537683,Sách Và Rồi Chẳng Còn Ai (Agatha Christie),154,151,3,98.05
10,50322710,Sách 48 Nguyên Tắc Chủ Chốt Của Quyền Lực (Tái...,334,332,2,99.40
25,271380890,Sách Đúng Việc,303,301,2,99.34


In [ ]:
product_stats.to_excel('Thong_Ke.xlsx', index=False)